In [1]:
import os
import torch
import torch.nn.functional as F
import numpy as np
import cv2
import matplotlib.pyplot as plt
from tqdm import tqdm

from lib.networks_prototype import Prototype_CASCADE
from utils.dataloader import test_dataset

def visualize_patches(cfg):
    # 1. 初始化
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    os.makedirs(cfg['output_dir'], exist_ok=True)
    
    print(f"Loading model from {cfg['model_path']}...")
    # 假設您的模型結構參數與訓練時一致
    model = Prototype_CASCADE(num_fg=cfg['num_fg'], num_bg=cfg['num_bg']).to(device)
    model.load_state_dict(torch.load(cfg['model_path'], map_location=device))
    model.eval()
    
    # 2. 準備數據集
    # test_dataset 需要 image_root 和 gt_root
    test_loader = test_dataset(cfg['image_root'], cfg['gt_root'], cfg['img_size'])
    num_images = test_loader.size
    print(f"Found {num_images} test images.")
    
    # 3. 收集 Patch 的容器
    # 我們需要為每個前景原型維護一個列表，存儲 (similarity_score, image_path, center_x, center_y)
    # 這裡我們用一個最小堆或者簡單的列表排序來保留 Top-K
    prototype_activations = {k: [] for k in range(cfg['num_fg'])}
    
    # 緩存圖像數據以便最後切片 (如果內存夠大) 或者只存路徑
    # 為了簡單，我們先遍歷一遍收集分數，再讀圖切片
    
    print("Scanning dataset for top activations...")
    with torch.no_grad():
        for i in tqdm(range(num_images)):
            image_tensor, gt, name = test_loader.load_data()
            image_tensor = image_tensor.to(device)
            
            # Forward
            # 假設模型返回 (logits, similarity_map)
            # similarity_map shape: (1, K_total, H, W)
            _, similarity_map = model(image_tensor)
            
            # 我們只關心前景原型
            fg_sim_map = similarity_map[0, :cfg['num_fg'], :, :] # (K_fg, H, W)
            
            # 獲取原圖尺寸 (用於坐標映射)
            # 注意：test_dataset 會 resize 圖像到 img_size，所以這裡 H, W 就是 img_size
            H, W = cfg['img_size'], cfg['img_size']
            
            # 對於每個前景原型，找到這張圖中激活度最高的一個點
            for k in range(cfg['num_fg']):
                sim_map_k = fg_sim_map[k] # (H, W)
                
                # 找到最大值及其位置
                max_val = sim_map_k.max().item()
                max_idx = sim_map_k.argmax().item()
                
                # 轉換為 (y, x)
                # 注意：similarity_map 的尺寸可能已經被上採樣到 img_size，也可能還是 feature map size
                # 根據您的模型代碼，它通常已經被 interpolate 到 img_size
                # 如果沒有，需要在這裡處理縮放
                
                h_feat, w_feat = sim_map_k.shape
                y_feat, x_feat = divmod(max_idx, w_feat)
                
                # 存儲信息
                # 我們存儲圖像索引 i，方便之後重新加載
                prototype_activations[k].append({
                    'score': max_val,
                    'img_idx': i,
                    'y': y_feat,
                    'x': x_feat,
                    'feat_h': h_feat,
                    'feat_w': w_feat
                })

    # 4. 篩選 Top-K 並生成可視化
    print("Generating patches...")
    patch_size = cfg['patch_size']
    top_k = cfg['top_k']
    
    for k in range(cfg['num_fg']):
        # 對該原型的所有激活按分數降序排序
        activations = sorted(prototype_activations[k], key=lambda x: x['score'], reverse=True)
        top_activations = activations[:top_k]
        
        # 創建一個畫布來拼貼 Patches
        # 假設我們把 Top-K 排成一行
        canvas = np.zeros((patch_size, patch_size * top_k, 3), dtype=np.uint8)
        
        for idx, act in enumerate(top_activations):
            # 重新加載圖像 (為了節省內存，我們這裡重新讀取)
            # 這裡有點低效，但對於小數據集可以接受。
            # 如果數據集很大，建議優化這裡 (例如在第一遍就緩存 Top-K 的圖像)
            
            # hack: 我們需要通過 index 訪問 test_loader 的數據，但 test_loader 設計是 iterator
            # 我們直接訪問其內部列表
            img_path = test_loader.images[act['img_idx']]
            img_pil = test_loader.rgb_loader(img_path)
            img_pil = img_pil.resize((cfg['img_size'], cfg['img_size'])) # 確保尺寸一致
            img_np = np.array(img_pil)
            
            # 計算中心點坐標 (映射回 img_size)
            scale_y = cfg['img_size'] / act['feat_h']
            scale_x = cfg['img_size'] / act['feat_w']
            
            center_y = int(act['y'] * scale_y)
            center_x = int(act['x'] * scale_x)
            
            # 截取 Patch
            y1 = max(0, center_y - patch_size // 2)
            y2 = min(cfg['img_size'], center_y + patch_size // 2)
            x1 = max(0, center_x - patch_size // 2)
            x2 = min(cfg['img_size'], center_x + patch_size // 2)
            
            patch = img_np[y1:y2, x1:x2, :]
            
            # 如果 Patch 在邊緣，可能小於 patch_size，需要填充
            h_p, w_p, _ = patch.shape
            pad_y = (patch_size - h_p) // 2
            pad_x = (patch_size - w_p) // 2
            
            # 簡單處理：直接貼到畫布對應位置
            target_y = 0 + pad_y if y1 == 0 else 0
            target_x = idx * patch_size + pad_x if x1 == 0 else idx * patch_size
            
            # 確保邊界不溢出
            target_h = min(patch_size, h_p)
            target_w = min(patch_size, w_p)
            
            canvas[0:target_h, idx * patch_size : idx * patch_size + target_w, :] = patch[:target_h, :target_w, :]
            
            # 在 Patch 上畫一個框或者分數 (可選)
            # cv2.putText(canvas, f"{act['score']:.2f}", (idx*patch_size+5, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 1)

        # 保存該原型的 Patch 圖
        save_path = os.path.join(cfg['output_dir'], f"prototype_{k+1}_top{top_k}.png")
        # OpenCV 使用 BGR，PIL 使用 RGB，轉換一下顏色
        cv2.imwrite(save_path, cv2.cvtColor(canvas, cv2.COLOR_RGB2BGR))
        print(f"Saved visualization for Prototype {k+1} to {save_path}")

if __name__ == '__main__':
    config = {
        'model_path': '/home/U116med/wch_code/cascade/models/prototype_cascade/baseline_prototype_fb_88/best.pth',
        'image_root': '/home/U116med/data/polyp/TestDataset/test/images/',
        'gt_root': '/home/U116med/data/polyp/TestDataset/test/masks/',
        
        'img_size': 352,
        'num_fg': 8,
        'num_bg': 8,
        
        'top_k': 10,       # 每個原型展示前 K 個最匹配的 Patch
        'patch_size': 64,  # Patch 的大小
        'output_dir': './vis_patches_88/'
    }
    
    visualize_patches(config)

/home/U116med/anaconda3/envs/ch/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/U116med/wch_code/cascade/lib/pvtv2.py:388: UserWarning: Overwriting pvt_v2_b0 in registry with lib.pvtv2.pvt_v2_b0. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  class pvt_v2_b0(PyramidVisionTransformerImpr):
/home/U116med/wch_code/cascade/lib/pvtv2.py:398: UserWarning: Overwriting pvt_v2_b1 in registry with lib.pvtv2.pvt_v2_b1. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  class pvt_v2_b1(PyramidVisionTransformerImpr):
/home/U116med/wch_code/cascade/lib/pvtv2.py:406: UserWarning: Overwriting pvt_v2_b2 in registry with lib.pvtv2.pvt_v2_b2. This is because the name being registered con

Loading model from /home/U116med/wch_code/cascade/models/prototype_cascade/baseline_prototype_fb_88/best.pth...
Model decoder:  created, param count: 1913515


/tmp/ipykernel_472587/686722424.py:20: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(cfg['model_path'], map_location=device))


Found 798 test images.
Scanning dataset for top activations...


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 798/798 [00:23<00:00, 34.19it/s]


Generating patches...
Saved visualization for Prototype 1 to ./vis_patches_88/prototype_1_top10.png
Saved visualization for Prototype 2 to ./vis_patches_88/prototype_2_top10.png
Saved visualization for Prototype 3 to ./vis_patches_88/prototype_3_top10.png
Saved visualization for Prototype 4 to ./vis_patches_88/prototype_4_top10.png
Saved visualization for Prototype 5 to ./vis_patches_88/prototype_5_top10.png
Saved visualization for Prototype 6 to ./vis_patches_88/prototype_6_top10.png
Saved visualization for Prototype 7 to ./vis_patches_88/prototype_7_top10.png
Saved visualization for Prototype 8 to ./vis_patches_88/prototype_8_top10.png
